In [ ]:
import os
import torch
from PIL import Image
from torchvision import transforms
from rembg import remove
import timm
import argparse
import cv2
import numpy as np
from pytorch_grad_cam import GradCAM, ScoreCAM, GradCAMPlusPlus, AblationCAM, XGradCAM, EigenCAM, EigenGradCAM, LayerCAM, FullGrad
from pytorch_grad_cam.utils.image import show_cam_on_image, preprocess_image
from pytorch_grad_cam.ablation_layer import AblationLayerVit

# デバイスの設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 画像の形式を統一する関数
def convert_image_format(file_path, output_dir):
    file_ext = os.path.splitext(file_path)[1].lower()
    if file_ext == '.heic':
        heif_file = pyheif.read(file_path)
        image = Image.frombytes(heif_file.mode, heif_file.size, heif_file.data, "raw", heif_file.mode, heif_file.stride)
        new_file_path = os.path.join(output_dir, os.path.splitext(os.path.basename(file_path))[0] + '.jpg')
    elif file_ext in ['.jpeg', '.jpg']:
        new_file_path = os.path.join(output_dir, os.path.splitext(os.path.basename(file_path))[0] + '.jpg')
        image = Image.open(file_path)
    else:
        raise ValueError("Unsupported file format")
    image.save(new_file_path, "JPEG")
    return new_file_path

# 背景を削除する関数
def remove_background(file_path, output_dir):
    with open(file_path, "rb") as f:
        img = f.read()
        output = remove(img)
    new_file_path = os.path.join(output_dir, os.path.basename(file_path))
    with open(new_file_path, "wb") as f_out:
        f_out.write(output)
    return new_file_path

# 画像の処理を行う関数
def process_images(right_hand_path, left_hand_path):
    output_dir_format = 'output/formatted'
    output_dir_bg_removed = 'output/bg_removed'
    os.makedirs(output_dir_format, exist_ok=True)
    os.makedirs(output_dir_bg_removed, exist_ok=True)

    right_hand_formatted = convert_image_format(right_hand_path, output_dir_format)
    left_hand_formatted = convert_image_format(left_hand_path, output_dir_format)

    right_hand_bg_removed = remove_background(right_hand_formatted, output_dir_bg_removed)
    left_hand_bg_removed = remove_background(left_hand_formatted, output_dir_bg_removed)

    return right_hand_bg_removed, left_hand_bg_removed

# 画像の前処理関数
def preprocess_image(image_path):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    image = Image.open(image_path).convert('RGB')
    return transform(image).unsqueeze(0)  # バッチ次元を追加

# モデルのロード関数
def load_model(model_path):
    model = timm.create_model('vit_base_patch16_224_in21k', pretrained=False, num_classes=2)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    return model

# 推論関数
def predict(model, image_tensor):
    with torch.no_grad():
        image_tensor = image_tensor.to(device)
        outputs = model(image_tensor)
        _, predicted = torch.max(outputs, 1)
        return predicted.item()

# ユーザーから提供された画像のパスを設定
right_hand_path = "path/to/right_hand_image.jpg"  # 右手画像のパスを指定
left_hand_path = "path/to/left_hand_image.jpg"    # 左手画像のパスを指定

# 学習済みモデルのパス
model_path = 'path/to/model.pth'  # 学習済みモデルのパスを指定

# 画像の処理
right_hand_processed, left_hand_processed = process_images(right_hand_path, left_hand_path)

# 画像の前処理
right_hand_tensor = preprocess_image(right_hand_processed)
left_hand_tensor = preprocess_image(left_hand_processed)

# モデルのロード
model = load_model(model_path)

# 予測
right_hand_prediction = predict(model, right_hand_tensor)
left_hand_prediction = predict(model, left_hand_tensor)

# 結果の表示
prediction_labels = ["Non-RA", "RA"]
print(f"Right hand prediction: {prediction_labels[right_hand_prediction]}")
print(f"Left hand prediction: {prediction_labels[left_hand_prediction]}")

# CAMの設定
def get_args():
    class Args:
        use_cuda = True
        image_path = '/content/drive/MyDrive/OptPhotoFiles/optPhotoRaRight/RA_Left_Hand_08.jpg'
        aug_smooth = False
        eigen_smooth = False
        method = 'gradcam'
    return Args()

args = get_args()

# 関数"reshape_transform"は、入力されたテンソルの幅と高さを変形する。デフォルトは14×14。
def reshape_transform(tensor, height=14, width=14):
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    result = result.transpose(2, 3).transpose(1, 2)
    return result

# スクリプトが直接実行された場合のみ、以下のコードブロックを実行する。
if __name__ == '__main__':
    # 利用可能なCAMメソッドの辞書を定義
    methods = {
        "gradcam": GradCAM,
        "scorecam": ScoreCAM,
        "gradcam++": GradCAMPlusPlus,
        "ablationcam": AblationCAM,
        "xgradcam": XGradCAM,
        "eigencam": EigenCAM,
        "eigengradcam": EigenGradCAM,
        "layercam": LayerCAM,
        "fullgrad": FullGrad
    }

    if args.method not in methods:
        raise Exception(f"method should be one of {list(methods.keys())}")

    # モデルの読み込みと初期化
    model = timm.create_model('vit_base_patch16_224.augreg_in21k', pretrained=False, num_classes=2)
    checkpoint = torch.load('/content/drive/MyDrive/OptPhotoFiles/model.pth')
    model.load_state_dict(checkpoint)
    model.eval()

    if args.use_cuda:
        model = model.cuda()

    target_layers = [model.blocks[-1].norm1]

    if args.method == "ablationcam":
        cam = methods[args.method](model=model, target_layers=target_layers, reshape_transform=reshape_transform, ablation_layer=AblationLayerVit())
    else:
        cam = methods[args.method](model=model, target_layers=target_layers, reshape_transform=reshape_transform)

    rgb_img = cv2.imread(args.image_path, 1)[:, :, ::-1]
    rgb_img = cv2.resize(rgb_img, (224, 224))
    rgb_img = np.float32(rgb_img) / 255
    input_tensor = preprocess_image(rgb_img, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

    targets = None
    cam.batch_size = 32

    grayscale_cam = cam(input_tensor=input_tensor, targets=targets, eigen_smooth=args.eigen_smooth, aug_smooth=args.aug_smooth)
    grayscale_cam = grayscale_cam[0, :]
    cam_image = show_cam_on_image(rgb_img, grayscale_cam)
    cv2.imwrite(f'{args.method}_cam.jpg', cam_image)

    import matplotlib.pyplot as plt
    plt.imshow(cam_image)
    plt.show()